In [6]:
import pandas as pd
import numpy as np
import yaml
import re
import os
from itertools import permutations
DB_FOLDER = f"dataset\database"
WINDOW_SIZE = 12
PRED_AHEAD = 1

In [7]:
from pathlib import Path

node_id = 10
timestep_dir = Path(DB_FOLDER) / "timestep"

price_history = []

for csv_path in sorted(timestep_dir.glob("*.csv"), key=lambda path: int(path.stem.split("_")[0])):
    timestep_df = pd.read_csv(csv_path)
    node_rows = timestep_df.loc[timestep_df["node_id"] == node_id].copy()

    if node_rows.empty:
        continue

    timestep_token, month_token = csv_path.stem.split("_", 1)
    node_rows["timestep_file"] = csv_path.name
    node_rows["lease_month"] = pd.to_datetime(f"{month_token}-01")
    node_rows["timestep_from_file"] = int(timestep_token)

    price_history.append(node_rows)

if price_history:
    price_history_df = (
        pd.concat(price_history, ignore_index=True)
        .sort_values(["timestep", "lease_month"])
        [["timestep", "node_id", "project_id", "rent_per_sqft", "y_mask"]]
        # .assign(timestep=lambda df: df["timestep"] - PRED_AHEAD)
        .reset_index(drop=True)
    )
else:
    price_history_df = pd.DataFrame(
        columns=[
            "timestep",
            "node_id",
            "project_id",
            "rent_per_sqft",
            "y_mask"
        ]
    )

price_history_df

,timestep,node_id,project_id,rent_per_sqft,y_mask
0,188,10,1.0,3.636364,1
1,189,10,1.0,3.727273,0
2,190,10,1.0,3.818182,1
3,191,10,1.0,3.272727,1
4,192,10,1.0,3.272727,1
...,...,...,...,...,...
96,284,10,1.0,6.772727,1
97,285,10,1.0,5.181818,1
98,286,10,1.0,5.772727,0
99,287,10,1.0,6.363636,1


In [8]:
price_history_df['timestep'] = price_history_df['timestep'] - 1
price_history_df

,timestep,node_id,project_id,rent_per_sqft,y_mask
0,187,10,1.0,3.636364,1
1,188,10,1.0,3.727273,0
2,189,10,1.0,3.818182,1
3,190,10,1.0,3.272727,1
4,191,10,1.0,3.272727,1
...,...,...,...,...,...
96,283,10,1.0,6.772727,1
97,284,10,1.0,5.181818,1
98,285,10,1.0,5.772727,0
99,286,10,1.0,6.363636,1


In [9]:
# make sure sorted
PAD_VALUE = np.nan   # or 0.0
df = price_history_df.sort_values("timestep").reset_index(drop=True)

X = []
X_timestep = []
X_mask = []

values = df["rent_per_sqft"].to_numpy()
timesteps = df["timestep"].to_numpy()

for i in range(len(df)):
    value_window = values[i:i + WINDOW_SIZE]
    timestep_window = timesteps[i:i + WINDOW_SIZE]

    valid_len = len(value_window)

    mask = np.zeros(WINDOW_SIZE, dtype=np.float32)
    mask[:valid_len] = 1.0

    if valid_len < WINDOW_SIZE:
        pad_len = WINDOW_SIZE - valid_len
        value_window = np.pad(value_window, (0, pad_len), constant_values=PAD_VALUE)
        timestep_window = np.pad(timestep_window, (0, pad_len), constant_values=-1)

    X.append(value_window)
    X_timestep.append(timestep_window)
    X_mask.append(mask)

X = np.array(X)                  # shape: (N, WINDOW_SIZE)
X_timestep = np.array(X_timestep)
X_mask = np.array(X_mask)

print("X shape:", X.shape)
print("X_timestep shape:", X_timestep.shape)
print("X_mask shape:", X_mask.shape)

X shape: (101, 12)
X_timestep shape: (101, 12)
X_mask shape: (101, 12)


In [10]:
# First 3 windows
print("=== First 3 Windows ===")
for i in range(3):
    print(f"\nWindow {i}")
    print("timesteps:", X_timestep[i])
    print("values   :", X[i])

# Last 3 windows
print("\n=== Last 3 Windows ===")
for i in range(len(X) - 3, len(X)):
    print(f"\nWindow {i}")
    print("timesteps:", X_timestep[i])
    print("values   :", X[i])

=== First 3 Windows ===

Window 0
timesteps: [187 188 189 190 191 192 193 194 195 196 197 198]
values   : [3.63636364 3.72727273 3.81818182 3.27272727 3.27272727 4.
 3.36363636 3.36363636 3.27272727 3.72727273 3.57575758 3.42424242]

Window 1
timesteps: [188 189 190 191 192 193 194 195 196 197 198 199]
values   : [3.72727273 3.81818182 3.27272727 3.27272727 4.         3.36363636
 3.36363636 3.27272727 3.72727273 3.57575758 3.42424242 3.27272727]

Window 2
timesteps: [189 190 191 192 193 194 195 196 197 198 199 200]
values   : [3.81818182 3.27272727 3.27272727 4.         3.36363636 3.36363636
 3.27272727 3.72727273 3.57575758 3.42424242 3.27272727 3.72727273]

=== Last 3 Windows ===

Window 98
timesteps: [285 286 287  -1  -1  -1  -1  -1  -1  -1  -1  -1]
values   : [5.77272727 6.36363636 7.09090909        nan        nan        nan
        nan        nan        nan        nan        nan        nan]

Window 99
timesteps: [286 287  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
values   : [6.36363